In [76]:
import pandas as pd
import os
import numpy as np

In [77]:
comb = []

for file in os.listdir("./raw-from-source/openelections-data-ms-master/2024/counties/"):
    temp = pd.read_csv("./raw-from-source/openelections-data-ms-master/2024/counties/"+file)
    comb.append(temp)

In [78]:
statewide = pd.concat(comb)

In [79]:
#statewide = statewide[statewide["office"]=="Court of Appeals"]

In [80]:
statewide.loc[statewide["votes"]=="X"]

,county,precinct,office,district,candidate,party,votes
182,Grenada,Mt. Nebo Young Water and Sewer,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
213,Tallahatchie,Blue Cane,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
220,Tallahatchie,Cascilla,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
222,Tallahatchie,Cascilla,Yazoo-Mississippi Delta Levee Board,NaN,"Benjamin ""Sykes"" Sturdivant",DEM,X
227,Tallahatchie,Charleston Beat #1,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
229,Tallahatchie,Charleston Beat #1,Yazoo-Mississippi Delta Levee Board,NaN,"Benjamin ""Sykes"" Sturdivant",DEM,X
234,Tallahatchie,Charleston Beat #2,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
241,Tallahatchie,Charleston Beat #3,Court of Appeals,2.0,Latrice Westbrooks,NaN,X
243,Tallahatchie,Charleston Beat #3,Yazoo-Mississippi Delta Levee Board,NaN,"Benjamin ""Sykes"" Sturdivant",DEM,X
248,Tallahatchie,Enid,Court of Appeals,2.0,Latrice Westbrooks,NaN,X


In [81]:
statewide.loc[statewide["votes"]=="X",'votes'] = 0

In [82]:
statewide["votes"] =statewide["votes"].astype(int)

In [83]:
statewide["votes"].unique()

array([  31,    1,    2, ..., 2990, 1061,  981], shape=(1674,))

In [84]:
statewide_pivot = pd.pivot_table(statewide, index = "county", columns = ["office","district","candidate"],values = "votes", aggfunc = "sum" )

In [85]:
statewide_pivot.reset_index(inplace = True)

In [86]:
statewide_pivot.to_csv("./rdh_coa_results.csv", index = False)

In [87]:
statewide = statewide[statewide["office"].isin(["President","Supreme Court","U.S. Senate","U.S. House"])]

In [88]:
statewide[statewide["office"]=="Supreme Court"][["county","precinct"]].value_counts()

county     precinct                   
Leake      East Carthage                  5
Kemper     Mount Nebo                     5
Jefferson  Red Lick                       5
           Stampley                       5
           Union Church                   5
                                         ..
Lee        Birmingham Ridge               2
           Belden                         2
           Beech Springs                  2
           Baldwyn                        2
Adams      Dist. 1, Bellemont Precinct    2
Name: count, Length: 1736, dtype: int64

In [89]:
statewide.loc[statewide["precinct"]=="Noxubee Human Resource Bldg","precinct"] = 'Noxubee Alliance Bldg'

In [90]:
statewide.loc[statewide["candidate"].isin(["Chase Oliver"]),"party"] = "LIB"
statewide.loc[statewide["candidate"].isin(["Claudia De la Cruz","Peter Sonski","Robert F. Kennedy Jr.","Shiva Ayyadurai"]),"party"] = "OTHER"

In [91]:
statewide.loc[:,"district"] = statewide.loc[:,"district"].fillna(9999)
statewide["office"] = np.where(statewide["district"]!=9999,statewide["office"]+"-"+statewide["district"].astype(float).astype(int).astype(str),statewide["office"])

In [92]:
statewide["candidate"] = statewide["candidate"].str.strip()

In [93]:
statewide.loc[:,"party"] = statewide.loc[:,"party"].fillna("OTHER")
statewide["contest_col"] = statewide["office"]+"-:-"+statewide["candidate"]+"-:-"+statewide["party"]

In [94]:
pivoted_results = pd.pivot_table(statewide, index = ["county","precinct"], columns = "contest_col", values = "votes")

In [95]:
pivoted_results

contest_col                            President-:-Chase Oliver-:-LIB  \
county precinct                                                         
Adams  Dist. 1, Bellemont Precinct                                2.0   
       Dist. 1, By-Pass Fire Precinct                             0.0   
       Dist. 1, Courthouse Precinct                               3.0   
       Dist. 2, Beau Pre Precinct                                 0.0   
       Dist. 2, Duncan Park Precinct                              4.0   
...                                                               ...   
Yazoo  Washington St. Fire Station                                0.0   
       Webster Street School                                      1.0   
       Welfare Office                                             0.0   
       West Bentonia                                              0.0   
       Zion                                                       0.0   

contest_col                            President-:-Claudia De la Cruz-:-OTHER  \
county precinct                                                                 
Adams  Dist. 1, Bellemont Precinct                                        1.0   
       Dist. 1, By-Pass Fire Precinct                                     1.0   
       Dist. 1, Courthouse Precinct                                       0.0   
       Dist. 2, Beau Pre Precinct                                         0.0   
       Dist. 2, Duncan Park Precinct                                      0.0   
...                                                                       ...   
Yazoo  Washington St. Fire Station                                        1.0   
       Webster Street School                                              2.0   
       Welfare Office                                                     1.0   
       West Bentonia                                                      1.0   
       Zion                                                               0.0   

contest_col                            President-:-Donald J. Trump-:-REP  \
county precinct                                                            
Adams  Dist. 1, Bellemont Precinct                                 850.0   
       Dist. 1, By-Pass Fire Precinct                              199.0   
       Dist. 1, Courthouse Precinct                                358.0   
       Dist. 2, Beau Pre Precinct                                  504.0   
       Dist. 2, Duncan Park Precinct                               397.0   
...                                                                  ...   
Yazoo  Washington St. Fire Station                                 162.0   
       Webster Street School                                        61.0   
       Welfare Office                                              102.0   
       West Bentonia                                               247.0   
       Zion                                                        201.0   

contest_col                            President-:-Jill Stein-:-GRN  \
county precinct                                                       
Adams  Dist. 1, Bellemont Precinct                              3.0   
       Dist. 1, By-Pass Fire Precinct                           2.0   
       Dist. 1, Courthouse Precinct                             1.0   
       Dist. 2, Beau Pre Precinct                               0.0   
       Dist. 2, Duncan Park Precinct                            2.0   
...                                                             ...   
Yazoo  Washington St. Fire Station                              0.0   
       Webster Street School                                    0.0   
       Welfare Office                                           0.0   
       West Bentonia                                            0.0   
       Zion                                                     0.0   

contest_col                            President-:-Kamala D. Harris-:-DEM  \
county

In [96]:
pivoted_results.reset_index(inplace = True, drop = False)

In [97]:
pivoted_results = pivoted_results.fillna(0)

In [98]:
for col in list(pivoted_results.columns):
    if col not in ["county","precinct"]:
        pivoted_results[col] = pivoted_results[col].astype(int)

In [99]:
def get_race(race_string):
    race_string = race_string.title()
    race_string = race_string.replace("(Vote For 1)","")
    if "U.S. House" in race_string or 'Us House' in race_string or "Representative To Congress" in race_string or "U. S. Representative" in race_string:
        return "CON"
    elif "State House" in race_string or "State Representative" in race_string:
        return "SL"
    elif "State Senate" in race_string or "State Senator" in race_string:
        return "SU"
    elif "President" in race_string:
        return "PRE"
    elif "US Senate" in race_string or "Us Senate" in race_string or "U.S. Senator" in race_string or "U. S. Senator" in race_string or "U.S. Senate" in race_string:
        return "USS"
    elif "Public Service" in race_string:
        return "PSC"
    elif "Attorney General" in race_string:
        return "ATG"
    elif "Auditor General" in race_string or "Auditor" in race_string:
        return "AUD"
    elif "Treasurer" in race_string:
        return "TRE"
    elif "Superintendent" in race_string:
        return "SUP"
    elif "Secretary Of State" in race_string:
        return "SOS"
    elif "Lieutenant Governor" in race_string:
        return "LTG"
    elif "Governor" in race_string:
        return "GOV"
    elif "Commissioner Of Labor" in race_string:
        return "LAB"
    elif "Commissioner Of Agriculture" in race_string:
        return "AGR"
    elif "Commissioner Of Insurance" in race_string:
        return "INS"
    elif "Chief Justice Of The Supreme Court" in race_string:
        return "CJU"
    elif "Justice Of The Supreme Court" in race_string:
        return "JUS"
    elif "Court Of Appeals" in race_string:
        return "COA"
    elif "Supreme Court" in race_string:
        return "SC"
    elif "Ca No." in race_string:
        num = race_string.split(" ")[4][:-1]
        return "A"+str(num)

    else:
        print("No race for:", race_string)
        raise ValueError
        
def get_election_type(race_string):
    if "CA No." in race_string:
        return ""
    else:
        return "G"
        
def get_party(race_string):
    if "Supreme Cour" in race_string:
        return "N"
    race_string = race_string.split("-:-")[-1]
    print("race string is", race_string)
    return race_string[0]
           
def get_name(name_string):
    if "No." in name_string:
        return ""
    else:
        #print(name_string)
        name_string = name_string.split(" (")[0]
        name_string = name_string.replace("'","")
        likely_last = name_string.split(" ")[-1]
        proposed_last = likely_last[:3]
        print(proposed_last)
        if proposed_last in ['II', 'III', 'Jr', 'Jr.', 'Sr.', 'JR.', "JR", "IV", 'I','Jr.',]:
            likely_last = name_string.split(" ")[-2]
            proposed_last = likely_last[:3]
        #print(proposed_last.upper())
        return proposed_last.upper()

import re
 

def get_district(race_string, fill_level):
    temp = race_string.split("-:-")[0].strip().split(" ")[-1]
    print(race_string)
    print(re.sub("\D","",temp).zfill(fill_level))
    return re.sub("\D","",temp).zfill(fill_level)

def column_rename_function(name_string):
    election_type = get_election_type(name_string)
    year = "24"
    party = get_party(name_string)
    race = get_race(name_string)
    district = ""
    #print(name_string)
    if race in ["CON", "SU"]:
        district = get_district(name_string, 2)
        year = ""
    elif race in ["SC"]:
        district = get_district(name_string, 1)
        year = ""
    name = get_name(name_string)
    new_col_name = election_type + year + race + district + party + name
#     print("election_type: ", election_type)
#     print("year: ", year)
#     print("race: ", race)
#     print("district: ", district)
#     print("party: ", party)
#     print("name: ", name)
    if len(new_col_name) > 10:
        print("LONG NAME")
        print(name_string, "->", new_col_name)
    return new_col_name

# Make a dictionary that points to the new column names and checks for duplicates

#la_20_state_level.rename(columns = {"Lessie Olivia Leblanc (DEM)-:-U. S. Representative -- 3rd Congressional District":"Lessie OliviaLeblanc (DEM)-:-U. S. Representative -- 3rd Congressional District"}, inplace = True)

races_to_clean = [i for i in pivoted_results.columns if i not in ["county","precinct"]]


race_updates_dict = {}
race_updates_reversed = {}
#clean_dups = {}
new_names = []
for val in races_to_clean:
    new_name = column_rename_function(val)
    race_updates_dict[val] = new_name
    if new_name not in new_names:
        new_names.append(new_name)
        race_updates_reversed[new_name] = val
    else:
        print("Duplicate", new_name)
        print(race_updates_reversed[new_name])
        print(val)
        #clean_dups[val] = race_updates_reversed[new_name]

race string is LIB
Oli
race string is OTHER
Cru
race string is REP
Tru
race string is GRN
Ste
race string is DEM
Har
race string is OTHER
Son
race string is CON
Ter
race string is OTHER
Jr.
race string is OTHER
Ayy
Supreme Court-1-:-Abby Gale Robinson-:-OTHER
1
Rob
Supreme Court-1-:-Byron Carter-:-OTHER
1
Car
Supreme Court-1-:-Ceola James-:-OTHER
1
Jam
Supreme Court-1-:-Jenifer B. Branning-:-OTHER
1
Bra
Supreme Court-1-:-Jim Kitchens-:-OTHER
1
Kit
Supreme Court-2-:-David P. Sullivan-:-OTHER
2
Sul
Supreme Court-2-:-Dawn H. Beam-:-OTHER
2
Bea
Supreme Court-3-:-Jimmy Maxwell-:-OTHER
3
Max
Supreme Court-3-:-Robert P. "Bobby" Chamberlin-:-OTHER
3
Cha
race string is DEM
U.S. House-1-:-Dianne Dodson Black-:-DEM
01
Bla
race string is REP
U.S. House-1-:-Trent Kelly-:-REP
01
Kel
race string is DEM
U.S. House-2-:-Bennie G. Thompson-:-DEM
02
Tho
race string is REP
U.S. House-2-:-Ron Eller-:-REP
02
Ell
race string is REP
U.S. House-3-:-Michael Guest-:-REP
03
Gue
race string is DEM
U.S. House-4-:-Cr

<>:87: SyntaxWarning: invalid escape sequence '\D'
<>:88: SyntaxWarning: invalid escape sequence '\D'
<>:87: SyntaxWarning: invalid escape sequence '\D'
<>:88: SyntaxWarning: invalid escape sequence '\D'
/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_41682/90293744.py:87: SyntaxWarning: invalid escape sequence '\D'
  print(re.sub("\D","",temp).zfill(fill_level))
/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_41682/90293744.py:88: SyntaxWarning: invalid escape sequence '\D'
  return re.sub("\D","",temp).zfill(fill_level)


Central District-Supreme Court District 1(Central) Position 3-:-Abby Gale RobinsonOTHER	Central District-Supreme Court District 1(Central) Position 3-:-Byron CarterOTHER	Central District-Supreme Court District 1(Central) Position 3-:-Ceola JamesOTHER	Central District-Supreme Court District 1(Central) Position 3-:-Jenifer B. BranningOTHER	Central District-Supreme Court District 1(Central) Position 3-:-Jim KitchensOTHER
Northern District-Supreme Court District 3(Northern) Position 1-:-Robert P. 'Bobby' ChamberlinOTHER
Northern District-Supreme Court District 3(Northern) Position 2-:-Jimmy Maxwell
OTHER
Southern District-Supreme Court District 2(Southern) Position 2-:-David P. SullivanOTHER
Southern District-Supreme Court District 2(Southern) Position 2-:-Dawn H. BeamOTHER

In [100]:
race_updates_dict['Supreme Court-1-:-Abby Gale Robinson-:-OTHER'] = 'GSC1P3NROB'
race_updates_dict['Supreme Court-1-:-Byron Carter-:-OTHER'] = 'GSC1P3NCAR'
race_updates_dict['Supreme Court-1-:-Ceola James-:-OTHER'] = 'GSC1P3NJAM'
race_updates_dict['Supreme Court-1-:-Jenifer B. Branning-:-OTHER'] = 'GSC1P3NBRA'
race_updates_dict['Supreme Court-1-:-Jim Kitchens-:-OTHER'] = 'GSC1P3NKIT'
race_updates_dict['Supreme Court-2-:-David P. Sullivan-:-OTHER'] = 'GSC2P2NSUL'
race_updates_dict['Supreme Court-2-:-Dawn H. Beam-:-OTHER'] = 'GSC2P2NBEA'
race_updates_dict['Supreme Court-3-:-Jimmy Maxwell-:-OTHER'] = 'GSC3P2NMAX'
race_updates_dict['Supreme Court-3-:-Robert P. "Bobby" Chamberlin-:-OTHER'] = 'GSC3P1NCHA'

In [101]:
race_updates_reversed = {j:i for i,j in race_updates_dict.items()}

In [102]:
comb_csv = pd.DataFrame(race_updates_reversed.items())
comb_csv.sort_values(0, inplace = True)
comb_csv.to_csv("./field_names.csv", index = False)

pivoted_results.rename(columns = race_updates_dict, inplace = True)

In [103]:
fips_file = pd.read_csv("./raw-from-source/FIPS/US_FIPS_Codes.csv")
fips_file = fips_file[fips_file["State"]=="Mississippi"]
fips_file["FIPS County"] = fips_file["FIPS County"].astype(str)
fips_file["FIPS County"] = fips_file["FIPS County"].str.zfill(3)
fips_file["County Name"] = fips_file["County Name"].replace("De Soto","Desoto")
fips_dict = dict(zip(fips_file['County Name'], fips_file['FIPS County']))
pivoted_results['COUNTYFP'] = pivoted_results['county'].map(fips_dict).fillna(pivoted_results['county'])
pivoted_results['COUNTYFP'] = pivoted_results['COUNTYFP'].astype(str).str.zfill(3)

# Print statements to check the county FIPs we've added
print(pivoted_results['COUNTYFP'].unique())

['001' '003' '005' '007' '009' '011' '013' '015' '017' '019' '021' '023'
 '025' '027' '029' '031' '033' '035' '037' '039' '041' '043' '045' '047'
 '049' '051' '053' '055' '057' '059' '061' '063' '065' '067' '069' '071'
 '073' '075' '077' '079' '081' '083' '085' '087' '089' '091' '093' '095'
 '097' '099' '101' '103' '105' '107' '109' '111' '113' '115' '117' '119'
 '121' '123' '125' '127' '129' '131' '133' '135' '137' '139' '141' '143'
 '145' '147' '149' '151' '153' '155' '157' '159' '161' '163']


In [104]:
pivoted_results["UNIQUE_ID"] = pivoted_results["county"] + "-:-" + pivoted_results["precinct"]

In [105]:
pivoted_results.rename(columns = {"county":"County","precinct":"Precinct"}, inplace = True)

In [106]:
# Sort the race columns in alphabetical order
race_cols = list(race_updates_reversed.keys())
race_cols.sort()

pivoted_results = pivoted_results[["UNIQUE_ID", "COUNTYFP","County", "Precinct"]+race_cols]

In [107]:
race_cols

['G24PRECTER',
 'G24PREDHAR',
 'G24PREGSTE',
 'G24PRELOLI',
 'G24PREOAYY',
 'G24PREOCRU',
 'G24PREOKEN',
 'G24PREOSON',
 'G24PRERTRU',
 'G24USSDPIN',
 'G24USSRWIC',
 'GCON01DBLA',
 'GCON01RKEL',
 'GCON02DTHO',
 'GCON02RELL',
 'GCON03RGUE',
 'GCON04DRAY',
 'GCON04REZE',
 'GSC1P3NBRA',
 'GSC1P3NCAR',
 'GSC1P3NJAM',
 'GSC1P3NKIT',
 'GSC1P3NROB',
 'GSC2P2NBEA',
 'GSC2P2NSUL',
 'GSC3P1NCHA',
 'GSC3P2NMAX']

In [108]:
pivoted_results.loc[pivoted_results["UNIQUE_ID"]=="Noxubee-:-Summerville","G24PREOSON"] = 0

In [113]:
for race in ['GSC3P1NCHA','GSC3P2NMAX','GCON01DBLA','GCON01RKEL']:
    print(pivoted_results.loc[pivoted_results["UNIQUE_ID"].isin(["Webster-:-Mathiston No. 4","Webster-:-Tomnolen","Webster-:-Walthall"]),race])

1677    273
1678    309
1679    295
Name: GSC3P1NCHA, dtype: int64
1677    275
1678    301
1679    298
Name: GSC3P2NMAX, dtype: int64
1677    17
1678    14
1679    15
Name: GCON01DBLA, dtype: int64
1677    297
1678    342
1679    322
Name: GCON01RKEL, dtype: int64


In [110]:
# tots = pd.DataFrame(pivoted_results.sum())
# results = pd.read_csv('/Users/peterhorton/Downloads/2024ElectionRecapSheets (2).csv')

# results["Party"] = results["Party"].fillna("OTHER")
# results["pivot_col"] = results["Office"]+"-:-"+results["Candidate"]+results["Party"]
# results_pivot = pd.pivot_table(results,index = 'County', columns = "pivot_col", values = "County Total")
# results_pivot = results_pivot.fillna(0)
# pivoted_results.groupby('County').sum().to_csv("./prec_state_totals.csv")
# results_pivot.to_csv("./official_state_totals.csv")

In [111]:
pivoted_results.dtypes

contest_col
UNIQUE_ID     object
COUNTYFP      object
County        object
Precinct      object
G24PRECTER     int64
G24PREDHAR     int64
G24PREGSTE     int64
G24PRELOLI     int64
G24PREOAYY     int64
G24PREOCRU     int64
G24PREOKEN     int64
G24PREOSON     int64
G24PRERTRU     int64
G24USSDPIN     int64
G24USSRWIC     int64
GCON01DBLA     int64
GCON01RKEL     int64
GCON02DTHO     int64
GCON02RELL     int64
GCON03RGUE     int64
GCON04DRAY     int64
GCON04REZE     int64
GSC1P3NBRA     int64
GSC1P3NCAR     int64
GSC1P3NJAM     int64
GSC1P3NKIT     int64
GSC1P3NROB     int64
GSC2P2NBEA     int64
GSC2P2NSUL     int64
GSC3P1NCHA     int64
GSC3P2NMAX     int64
dtype: object

In [112]:
if not os.path.exists("./ms_2024_gen_prec_csv"):
    os.mkdir("./ms_2024_gen_prec_csv")
    
pivoted_results.to_csv("./ms_2024_gen_prec_csv/ms_2024_gen_prec_csv.csv",index=False)

In [112]:
import shutil


In [113]:
shutil.make_archive("./ms_2024_gen_prec_csv","zip","./ms_2024_gen_prec_csv")

'/Users/peterhorton/Documents/RDH/pber_local/MS_2024/ms_2024_gen_prec_csv.zip'